In [9]:
# SparkSession — brama do Sparka
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [10]:
# Wczytanie pliku JSON
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [11]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [12]:
#Konwersja kolumny timestamp
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [13]:
# Liczba transakcji i suma przychodów per sklep
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [14]:
# Statystyki per kategoria
from pyspark.sql.functions import min as _min, max as _max

store_summary1 = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("category")
)
store_summary1.show()

+-----------+---------+----------+-----------+
|   category|liczba_tx|  suma_PLN|srednia_PLN|
+-----------+---------+----------+-----------+
|elektronika|     2542|1520770.69|     598.26|
|    książki|     2574| 851382.08|     330.76|
|     odzież|     2453| 849877.55|     346.46|
|    żywność|     2431| 789514.43|     324.77|
+-----------+---------+----------+-----------+



In [15]:
# Liczba transakcji per godzina (tumbling 1h)
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [16]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [17]:
# Okna 30-minutowe per sklep

hourly1 = (
    df.groupBy(window("timestamp", "30 minutes"), "store")    # okno 30-minutowe
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window","store")
)
hourly1.show(truncate=False)


+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:3

In [18]:
# W której godzinie sklep “Kraków” miał najwyższy przychód?
from pyspark.sql.functions import desc

hourly2 = (
    df.filter(col("store") == "Kraków") # wyfiltrowanie tylko dla miasta Kraków
    .groupBy("store", window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(desc("suma_PLN"))
)
hourly2.show(truncate=False)


+------+------------------------------------------+---------+---------+
|store |window                                    |liczba_tx|suma_PLN |
+------+------------------------------------------+---------+---------+
|Kraków|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|1169     |483309.86|
|Kraków|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|821      |341327.83|
|Kraków|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|532      |201259.26|
+------+------------------------------------------+---------+---------+



In [19]:
# Okna sliding - okno 1h, krok 30 minut
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [20]:
# Porównaj tumbling vs sliding
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ: ponieważ okna nakładają się na siebie, więc przedziały nie są rozłączne 

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") agreguje zdarzenia po sklepach, biorąc pod uwagę całą historię transakcji
#               a groupBy(window(),"store") agreguje zdarzenia po sklepach w podziale na okna czasowe (czyli np. w trakcie każdej godziny) 

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna zawiera te transakcje: 9:00-10:00, 9:30-10:30

In [21]:
#Praca domowa
# 1. Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
from pyspark.sql.functions import asc
hourly3 = (
    df.filter(col("store") == "Gdańsk") # wyfiltrowanie tylko dla miasta Kraków
    .groupBy("store", window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(asc("suma_PLN"))
)
hourly3.show(truncate=False)

+------+------------------------------------------+---------+---------+
|store |window                                    |liczba_tx|suma_PLN |
+------+------------------------------------------+---------+---------+
|Gdańsk|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|558      |230407.7 |
|Gdańsk|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|766      |302579.07|
|Gdańsk|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|1174     |488279.58|
+------+------------------------------------------+---------+---------+



In [25]:
# 2. Policz ile transakcji per kategoria było w oknie 09:00–09:30.
hourly4 = (
    df.filter((col("timestamp") >= "2026-04-12 09:00:00") & (col("timestamp") < "2026-04-12 09:30:00"))
    .groupBy(window("timestamp", "30 minutes"), "category")    # okno 30-minutowe
    .agg(
        count("tx_id").alias("liczba_tx"),
    )
    .orderBy("category")
)
hourly4.show(truncate=False)

+------------------------------------------+-----------+---------+
|window                                    |category   |liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |
+------------------------------------------+-----------+---------+



In [26]:
# 3. Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).
hourly5 = (
    df.groupBy(window("timestamp", "15 minutes"))    # okno 15-minutowe
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy(desc("liczba_tx"))
)
hourly5.show(truncate=False)

+------------------------------------------+---------+---------+
|window                                    |liczba_tx|suma_PLN |
+------------------------------------------+---------+---------+
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|1234     |481566.97|
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|1171     |440715.14|
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|1156     |504943.74|
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|1139     |475251.18|
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|1100     |469004.36|
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|899      |355500.31|
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|858      |359254.89|
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|644      |213061.19|
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|582      |224438.4 |
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|468      |198098.62|
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|443      |156900.51|
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|306      |132809.44|
+------------------------